# TB-Trust — 02: Train models (Phase 3)

Trains the in-distribution reference, both leave-one-clinic-out folds, and the TB-Net reproduction, then measures each architecture's cost.

In [ ]:
# --- configuration ---------------------------------------------------------
# Defaults are the Kaggle paths. Every path is read from the environment first,
# so the same notebook runs unmodified on Kaggle, locally, or in CI -- which is
# also what lets these notebooks be executed as a test rather than only read.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(WORK, exist_ok=True)
print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running a notebook is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()
print("tbtrust ready from", REPO)

In [ ]:
import subprocess
import sys


def run(cmd):
    """Run a CLI step and fail loudly rather than leaving a half-finished pipeline."""
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print((r.stdout or "")[-2500:])
    if r.returncode != 0:
        print((r.stderr or "")[-3000:])
        raise RuntimeError(f"command failed: {' '.join(str(c) for c in cmd)}")


TRAIN = [sys.executable, "-m", "tbtrust.train.loop"]
EVAL = [sys.executable, "-m", "tbtrust.eval.run"]

## 1. In-distribution reference

`configs/baseline_densenet.yaml` sets `split_mode: random`, so the test set is drawn from the same clinics as training. That is the "best case" number the cross-site gap is measured against.

It has to be **evaluated**, not read off training: `best_val_accuracy` is the score the checkpoint was selected on — a maximum over epochs — and using it as the reference would inflate the generalization gap by exactly that selection bias.

In [ ]:
run([*TRAIN, "--config", "configs/baseline_densenet.yaml",
             f"data.manifest={MANIFEST}", f"train.output_dir={OUT}/reference"])
run([*EVAL, "--config", "configs/baseline_densenet.yaml",
            "--checkpoint", f"{OUT}/reference/montgomery/best.ckpt",
            f"data.manifest={MANIFEST}"])

In [ ]:
import json
from pathlib import Path

ref = json.loads(Path(f"{OUT}/reference/montgomery/metrics.json").read_text())
assert ref["split_mode"] == "random", "the reference run must not be a LOCO fold"
REFERENCE_ACCURACY = ref["robustness_sweep"][str(ref.get("primary_severity", 0.5))]["accuracy"] \
    if str(ref.get("primary_severity", 0.5)) in ref["robustness_sweep"] \
    else ref["robustness_sweep"]["0.5"]["accuracy"]
print("in-distribution reference accuracy (held-out random-split test):", round(REFERENCE_ACCURACY, 4))
Path(f"{WORK}/reference_accuracy.json").write_text(json.dumps({"reference_accuracy": REFERENCE_ACCURACY}))

## 2. Leave-one-clinic-out folds

Montgomery and Shenzhen are the two-class holdouts. Each fold trains on the other clinics and is tested on a site it has never seen.

In [ ]:
for clinic in ["montgomery", "shenzhen"]:
    run([*TRAIN, "--config", f"configs/loco_{clinic}.yaml",
                 f"data.manifest={MANIFEST}", f"train.output_dir={OUT}/baseline"])
    run([*EVAL, "--config", f"configs/loco_{clinic}.yaml",
                "--checkpoint", f"{OUT}/baseline/{clinic}/best.ckpt",
                f"data.manifest={MANIFEST}"])

## 3. TB-Net reproduction

Path B: an attention-condenser CNN tuned to TB-Net's reported ~4.24M parameters. It is a reimplementation, not a port of the released TensorFlow checkpoint — any comparison to TB-Net's published 99.86% must say so.

In [ ]:
run([*TRAIN, "--config", "configs/tbnet_montgomery.yaml",
             f"data.manifest={MANIFEST}", f"train.output_dir={OUT}/tbnet"])
run([*EVAL, "--config", "configs/tbnet_montgomery.yaml",
            "--checkpoint", f"{OUT}/tbnet/montgomery/best.ckpt",
            f"data.manifest={MANIFEST}"])

## 4. Efficiency: params, MACs, CPU latency

Backs the low-compute claim with measurements. Single CPU thread approximates a low-end clinic device.

In [ ]:
run([sys.executable, "scripts/benchmark_efficiency.py",
     "--models", "baseline_densenet121,tbnet,evidential",
     "--threads", "1", "--repeats", "10",
     "--out", f"{WORK}/efficiency_benchmark.json"])

Next: **03_uncertainty_and_deferral.ipynb**.